# DATA GENERATION

In [1]:
import os
from google.cloud import bigquery
from faker import Faker
import pandas as pd
import random
from datetime import timedelta


PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

assert PROJECT_ID is not None, "Falta PROJECT_ID"
assert DATASET_ID is not None, "Falta DATASET_ID"

client = bigquery.Client(project=PROJECT_ID, location="EU")

fake = Faker()

BASE_ID = 10000


c:\Users\user\Documents\GitHub\tc-sql-Innovaciber\venv\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Clientes

In [2]:
customers = []

for i in range(500):
    customers.append({
        "customer_id": BASE_ID + i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "country": fake.country_code(),
        "city": fake.city(),
        "adquisition_channel": random.choice(["organic","paid_ads","social","referral"]),
        "registration_date": fake.date_time_between(start_date='-2y', end_date='now')
    })

df_customers = pd.DataFrame(customers)


## Categories

In [3]:
categories = [
    {"category_id": 1, "name": "Electronics", "description": "Electronic devices and accessories"},
    {"category_id": 2, "name": "Clothing", "description": "Fashion and apparel"},
    {"category_id": 3, "name": "Home", "description": "Home and kitchen products"},
    {"category_id": 4, "name": "Sports", "description": "Sports equipment and accessories"},
    {"category_id": 5, "name": "Books", "description": "Books and magazines"},
    {"category_id": 6, "name": "Beauty", "description": "Beauty and personal care"},
    {"category_id": 7, "name": "Toys", "description": "Toys and games"},
    {"category_id": 8, "name": "Automotive", "description": "Automotive accessories"},
    {"category_id": 9, "name": "Garden", "description": "Garden and outdoor"},
    {"category_id": 10, "name": "Food", "description": "Food and beverages"}
]

df_categories = pd.DataFrame(categories)
df_categories

,category_id,name,description
0,1,Electronics,Electronic devices and accessories
1,2,Clothing,Fashion and apparel
2,3,Home,Home and kitchen products
3,4,Sports,Sports equipment and accessories
4,5,Books,Books and magazines
5,6,Beauty,Beauty and personal care
6,7,Toys,Toys and games
7,8,Automotive,Automotive accessories
8,9,Garden,Garden and outdoor
9,10,Food,Food and beverages


## Productos

In [4]:
products = []

for i in range(70):
    price = round(random.uniform(10, 300), 2)

    products.append({
        "product_id": BASE_ID + i,
        "category_id": random.randint(1,10),
        "product_name": fake.word().capitalize(),
        "brand": fake.company(),
        "sale_price": price,
        "cost_price": round(price * random.uniform(0.4, 0.7), 2),
        "stock": random.randint(20,200),
        "is_active": True
    })

df_products = pd.DataFrame(products)


## Orders + Items

In [5]:
orders = []
order_items = []

order_id = BASE_ID
order_item_id = BASE_ID

for i in range(2000):
    order_date = fake.date_time_between(start_date='-1y', end_date='now')

    orders.append({
        "order_id": order_id,
        "customer_id": random.choice(df_customers["customer_id"]),
        "order_date": order_date,
        "order_status": "delivered"
    })

    for _ in range(random.randint(2,3)):
        product = random.choice(products)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order_id,
            "product_id": product["product_id"],
            "quantity": random.randint(1,4),
            "unit_price": product["sale_price"]
        })

        order_item_id += 1

    order_id += 1

df_orders = pd.DataFrame(orders)
df_order_items = pd.DataFrame(order_items)


## Pagos

In [6]:
payments = []

for o in orders:
    total = sum([
        oi["quantity"] * oi["unit_price"]
        for oi in order_items if oi["order_id"] == o["order_id"]
    ])

    payments.append({
        "payment_id": o["order_id"],
        "order_id": o["order_id"],
        "payment_method": random.choice(["credit_card","paypal","bank_transfer"]),
        "payment_status": "completed",
        "payment_date": o["order_date"],
        "amount": round(total,2)
    })

df_payments = pd.DataFrame(payments)


## Shipments

In [7]:
shipments = []

for o in orders:
    shipments.append({
        "shipments_id": o["order_id"],
        "order_id": o["order_id"],
        "shipping_status": "delivered",
        "shipping_country": fake.country_code(),
        "shipping_city": fake.city(),
        "delivery_date": o["order_date"] + timedelta(days=random.randint(1,5)),
        "street_address": fake.street_address(),
        "cp_code": fake.postcode(),
        "phone": fake.phone_number()
    })

df_shipments = pd.DataFrame(shipments)


## Reviews

In [8]:
reviews = []
review_id = BASE_ID

for oi in order_items:
    if random.random() < 0.35:
        reviews.append({
            "review_id": review_id,
            "customer_id": random.choice(df_customers["customer_id"]),
            "product_id": oi["product_id"],
            "rating": random.randint(1,5),
            "review_date": fake.date_time_between(start_date='-6m', end_date='now'),
            "comment": fake.sentence()
        })
        review_id += 1

df_reviews = pd.DataFrame(reviews)


## Upload

In [9]:
from google.cloud import bigquery

def upload(df, table_name):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_TRUNCATE"
        )
    )

    job.result()
    print(f"{table_name} añadida correctamente")

upload(df_categories, "categories")
upload(df_customers, "customers")
upload(df_products, "products")
upload(df_orders, "orders")
upload(df_order_items, "order_items")
upload(df_payments, "payments")
upload(df_shipments, "shipments")
upload(df_reviews, "reviews")


categories añadida correctamente
customers añadida correctamente
products añadida correctamente
orders añadida correctamente
order_items añadida correctamente
payments añadida correctamente
shipments añadida correctamente
reviews añadida correctamente
